# 03 - EDA (descriptive + associations; no causal claims)

In [ ]:
import os, sys, pathlib
MODE = "real"            # "sample" shows the SYNTHETIC test run (clearly watermarked). Never mix the two.
os.environ["DATA_MODE"] = MODE
# os.environ["START_MONTH"] = "2024-06"; os.environ["END_MONTH"] = "2024-06"
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import json, duckdb, pandas as pd
from IPython.display import Image, Markdown, HTML, display
import config, provenance
print(provenance.banner())
if config.IS_SYNTHETIC:
    display(HTML("<h3 style='color:red'>SYNTHETIC DATA - pipeline test only, not Citi Bike findings</h3>"))
man = provenance.load()
assert man is not None, "No manifest for this run yet - run the earlier phases first."

In [ ]:
if not (config.REP_DATA / 'daily.parquet').exists():
    import eda
Markdown((config.REPORT_DIR / 'EDA_insights.md').read_text())

## Volume & distributions

In [ ]:
display(Image(str(config.FIG/'01_daily_trips.png'))); display(Image(str(config.FIG/'02_distributions.png')))

**How to read this (guide, not a finding):** look for trend vs seasonality and holiday/weekday dips; heavy right tail in duration (log scale); speed spike near 0 = bike not moved / round trips.

## Time patterns

In [ ]:
display(Image(str(config.FIG/'03_hourly_weekday_weekend.png'))); display(Image(str(config.FIG/'04_heatmap_dow_hour.png')))

**How to read this (guide, not a finding):** curves are per-day averages so weekday and weekend are comparable; two weekday peaks suggest commuting, one midday weekend hump suggests leisure. The heatmap shows raw counts (weekdays outnumber weekend days).

## Riders

In [ ]:
display(Image(str(config.FIG/'05_member_vs_casual.png'))); pd.read_csv(config.REP_DATA/'user_summary.csv')

## Stations

In [ ]:
for i in ['06_top_start_stations','08_station_imbalance']: display(Image(str(config.FIG/f'{i}.png')))

**How to read this (guide, not a finding):** imbalance is a ratio among above-median-activity stations and reflects trip counts, not bike inventory (rebalancing trucks move bikes unseen).

## Weather - ASSOCIATIONS

In [ ]:
display(Image(str(config.FIG/'10b_weather_relative_demand.png')))
if (config.REP_DATA/'weather_adjusted.csv').exists(): display(pd.read_csv(config.REP_DATA/'weather_adjusted.csv').round(3))

**How to read this (guide, not a finding):** bars compare an hour with the same month x weekday x hour norm (ratio 1 = typical). The regression adjusts for weekend, holiday and a linear trend with HAC errors. Over a short window temperature, season and growth cannot be separated, so read effects as associations; if `reliability` says exploratory, do not quote the numbers.

In [ ]:
display(Image(str(config.FIG/'10_weather_unadjusted.png')))

**How to read this (guide, not a finding):** the unadjusted panels are confounded (season, weekday) and are shown for transparency only.

In [ ]:
display(Image(str(config.FIG/'11_correlation_matrix.png')))